<a href="https://colab.research.google.com/github/Naylet92/Estudio_ambiental/blob/main/Quila_L_2000.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DESCRIPCIÓN DEL PROCESAMIENTO DE IMÁGENES LANDSAT 5(AÑO 2000)

##1. Información general de las imágenes
Para el análisis multitemporal se utilizaron imágenes de la colección Landsat 5 TM Collection 2 Level-2 (reflectancia superficial) correspondientes al año 2000. Se definieron dos temporadas de estudio:

Temporada seca: 1 de marzo al 31 de mayo-----> Evaluar condiciones de estrés hídrico y vegetación en estiaje

Temporada húmeda: 1 de agosto al 30 de noviembre-----> 	Evaluar el máximo desarrollo vegetal post-lluvias

El área de estudio corresponde a la Sierra de Quila, delimitada por un polígono vectorial cargado desde un archivo GeoJSON. Todas las imágenes fueron recortadas a esta zona de interés.

##2. Procesamiento aplicado
El flujo de procesamiento incluyó las siguientes etapas:

2.1. Enmascaramiento de nubes y sombras
Se utilizó la banda de calidad QA_PIXEL para excluir píxeles afectados por nubes y sombras, asegurando que solo información válida contribuyera al mosaico final.

2.2. Corrección topográfica
Se aplicó una corrección por iluminación topográfica utilizando el modelo digital de elevación SRTM (30 m). El método corrige las diferencias de iluminación causadas por la pendiente y orientación del terreno, normalizando las reflectancias con el ángulo solar y el coseno del ángulo de incidencia.

2.3. Composición de mediana
Para cada temporada se generó un mosaico de mediana utilizando todas las imágenes disponibles. Este enfoque reduce el ruido y minimiza la influencia de artefactos residuales de nubes o sombras.

2.4. Cálculo de índices y componentes
A partir del mosaico de mediana se derivaron:

NDVI (Índice de Vegetación de Diferencia Normalizada)
NDVI = (NIR - Rojo) / (NIR + Rojo)

Componentes de Tasseled Cap (Brightness, Greenness, Wetness)
Utilizando los coeficientes estándar para Landsat 5 TM (Crist, 1985), que permiten separar información estructural, de verdor y de humedad.
| Componente | B1 (Azul) | B2 (Verde) | B3 (Rojo) | B4 (NIR) | B5 (SWIR1) | B7 (SWIR2) |
|------------|-----------|------------|-----------|----------|------------|------------|
| **Brightness** | 0.3037 | 0.2793 | 0.4743 | 0.5585 | 0.5082 | 0.1863 |
| **Greenness** | -0.2848 | -0.2435 | -0.5436 | 0.7243 | 0.0840 | -0.1800 |
| **Wetness** | 0.1509 | 0.1973 | 0.3279 | 0.3406 | -0.7112 | -0.4572 |

##3. Imágenes combinadas por temporada
Temporada seca (8 imágenes)
Se combinaron las siguientes escenas del año 2000:

| 🌦️ Temporada | 📅 Rango de fechas | 📷 Imágenes | 🗓️ Fechas de adquisición |
|-------------|-------------------|-------------|-------------------------|
| **Seca** | 01/03 – 31/05 | 8 | `2000-04-03` · `2000-04-10` · `2000-04-19` · `2000-04-26` · `2000-05-05` · `2000-05-12` · `2000-05-21` · `2000-05-28` |
| **Húmeda** | 01/08 – 30/11 | 11 | `2000-08-09` · `2000-08-16` · `2000-08-25` · `2000-09-01` · `2000-09-10` · `2000-09-17` · `2000-09-26` · `2000-10-03` · `2000-10-12` · `2000-10-19` · `2000-11-29` |



##4. Parámetros de visualización obtenidos
Los valores estadísticos de las bandas resultantes fueron:
| Variable | Temporada | Mínimo | Máximo | Paleta | Interpretación del rango |
|----------|-----------|--------|--------|--------|--------------------------|
| **NDVI** | Seca | 0.10 | 0.79 | Rojo → Amarillo → Verde | Desde suelo desnudo hasta vegetación densa |
| | Húmeda | 0.12 | 0.87 | Rojo → Amarillo → Verde | Mayor vigor vegetal en el extremo superior |
| **TC Brightness** | Seca | 0.00 | 1.40 | Blanco → Café | Contraste entre sombras y superficies brillantes |
| | Húmeda | 0.00 | 1.39 | Blanco → Café | Rango similar a temporada seca |
| **TC Greenness** | Seca | -0.03 | 0.33 | Blanco → Verde oscuro | Vegetación presente pero limitada |
| | Húmeda | -0.01 | 0.84 | Blanco → Verde oscuro | **Mucha más vegetación activa** |
| **TC Wetness** | Seca | -0.50 | 0.03 | Café → Blanco → Azul | Condiciones muy secas (valores negativos) |
| | Húmeda | -0.27 | 0.08 | Café → Blanco → Azul | Mayor humedad superficial |

## Selección y procesamiento de imágenes Landsat (año 2000)


##Definir Variables

In [ ]:
# Prefijo del área
PREFIJO     = 'QLA'
# Nombre del área
NOMBRE      = 'Sierra_Quila'

# Carpeta de salida en Drive
RUTA_AOI    = '/content/drive/MyDrive/ANP/AOI'

# Rutas
RUTA        = f'{RUTA_AOI}/aoi_{PREFIJO}.geojson'
RUTA_GEOJSON= RUTA
RUTA_CSV    = f'{RUTA_AOI}/{PREFIJO}_coordenadas.csv'

# CRS geográfico y UTM
CRS_GEO     = 4326
CRS_UTM     = 32613

# Proyecto de Google Earth Engine
GEE_PROJECT = 'ee-nayleths'

AÑO           = 2000
COLECCION     = 'LANDSAT/LT05/C02/T1_L2'

# Temporadas de análisis
TEMPORADA_SECA   = ('03-01', '05-31')
TEMPORADA_HUMEDA = ('08-01', '11-30')

# Bandas espectrales de reflectancia superficial
BANDAS_SR     = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
BANDAS_CORR   = [b + '_corr' for b in BANDAS_SR]

# Coeficientes Tasseled Cap
TC_BRIGHT = [0.3037, 0.2793, 0.4743, 0.5585, 0.5082, 0.1863]
TC_GREEN  = [-0.2848, -0.2435, -0.5436, 0.7243, 0.0840, -0.1800]
TC_WET    = [0.1509, 0.1973, 0.3279, 0.3406, -0.7112, -0.4572]

# Escala de exportación (m)
ESCALA        = 30

# Zoom inicial del mapa
ZOOM        = 12

# Estilo del área de estudio
STYLE_AREA  = {'color': 'blue', 'fillColor': '#0000ff30', 'weight': 1.5}

# Estilo del rectángulo rojo o bounding box
STYLE_BBOX  = {'color': 'red', 'fillColor': '#00000000', 'weight': 2.5}

# Título del mapa
MAP_TITLE   = 'Área de estudio'

# Carpeta de salida en Drive para imágenes GEE
RUTA_IMAGENES = f'/content/drive/MyDrive/ANP/Landsat'

In [ ]:
import ee
import os
import math
import json
import geemap
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from google.colab import drive
from IPython.display import display, HTML

# Montar Google Drive
drive.mount('/content/drive')

# Autenticar e inicializar GEE
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

print("✓ Entorno listo")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Entorno listo


In [ ]:
# Enmascarar nubes y sombras
def mask_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow_mask = qa.bitwiseAnd(1 << 3).eq(0)
    cloud_mask        = qa.bitwiseAnd(1 << 4).eq(0)
    return image.updateMask(cloud_shadow_mask.And(cloud_mask))


def correccion_topografica(image, region):
    dem        = ee.Image('USGS/SRTMGL1_003').clip(region)
    terreno    = ee.Terrain.products(dem)

    pendiente   = terreno.select('slope').multiply(math.pi / 180)
    orientacion = terreno.select('aspect').multiply(math.pi / 180)

    az_solar   = ee.Number(image.get('SUN_AZIMUTH')).multiply(math.pi / 180)
    elev_solar = ee.Number(image.get('SUN_ELEVATION')).multiply(math.pi / 180)

    cos_i = (
        pendiente.cos().multiply(elev_solar.sin())
        .add(
            pendiente.sin()
            .multiply(elev_solar.cos())
            .multiply(orientacion.subtract(az_solar).cos())
        )
    )
    cos_i_safe = cos_i.max(0.2)

    # Factor de escala para Collection 2 (DN * 0.0000275 - 0.2)
    im_bandas = image.select(BANDAS_SR).multiply(0.0000275).add(-0.2)

    bandas_corr = (
    ee.Image.cat([
        im_bandas.select(b).divide(cos_i_safe).rename(BANDAS_CORR[i])
        for i, b in enumerate(BANDAS_SR)
        ])
        .set('SUN_AZIMUTH',   image.get('SUN_AZIMUTH'))
        .set('SUN_ELEVATION', image.get('SUN_ELEVATION'))
        .cast({b: 'double' for b in BANDAS_CORR})
    )

    return bandas_corr


def tasseled_cap(image_corr, coefs, nombre):
    partes = ee.Image.cat([
        image_corr.select(b).multiply(c)
        for b, c in zip(BANDAS_CORR, coefs)
    ])
    return partes.reduce(ee.Reducer.sum()).rename(nombre)

### Procesamiento por temporada

In [ ]:

# Región GEE desde el GeoJSON generado anteriormente
gdf = gpd.read_file(RUTA_GEOJSON)
geojson_str = gdf.to_crs(epsg=4326).to_json()
region = ee.FeatureCollection(json.loads(geojson_str)).geometry()

#Procesar temporada Seca y de Húmedad
def procesar_temporada(nombre_temporada, fecha_ini, fecha_fin):
    print(f"\n── Temporada: {nombre_temporada} ({fecha_ini} → {fecha_fin}) ──")

    fecha_inicio = f'{AÑO}-{fecha_ini}'
    fecha_fin_   = f'{AÑO}-{fecha_fin}'

    col = (ee.ImageCollection(COLECCION)
             .filterDate(fecha_inicio, fecha_fin_)
             .filterBounds(region))

    n = col.size().getInfo()
    print(f"  Imágenes encontradas: {n}")

    fechas = col.aggregate_array('DATE_ACQUIRED').getInfo()
    for f in sorted(fechas):
        print(f"    · {f}")

    if n == 0:
        print(" Sin imágenes para esta temporada.")
        return None

    # 1. Enmascarar las nubes por imagen
    col_limpia = col.map(mask_clouds)

    # 2. Corrección topográfica por imagen
    col_corr = col_limpia.map(lambda img: correccion_topografica(img, region))

    # 3. Forzar bandas homogéneas antes del mosaico
    col_corr = col_corr.select(BANDAS_CORR)

    # 4. Mosaico de mediana
    mosaico = col_corr.median().clip(region)

    # 5. NDVI
    nir  = mosaico.select('SR_B4_corr')
    red  = mosaico.select('SR_B3_corr')
    ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')

    # 6. Tasseled Cap
    tc_b = tasseled_cap(mosaico, TC_BRIGHT, 'TC_Brightness')
    tc_g = tasseled_cap(mosaico, TC_GREEN,  'TC_Greenness')
    tc_w = tasseled_cap(mosaico, TC_WET,    'TC_Wetness')

    resultado = (ee.Image.cat([ndvi, tc_b, tc_g, tc_w])
                   .clip(region)
                   .cast({'NDVI': 'double', 'TC_Brightness': 'double',
                          'TC_Greenness': 'double', 'TC_Wetness': 'double'}))
    print(f"  Bandas resultantes: {resultado.bandNames().getInfo()}")
    return resultado


### Exportar resultados a Google Drive

In [ ]:

#  Ejecutar procesamiento
img_seca   = procesar_temporada('Seca',   *TEMPORADA_SECA)
img_humeda = procesar_temporada('Húmeda', *TEMPORADA_HUMEDA)

# Exportar a Google Drive
def exportar(imagen, sufijo):
    if imagen is None:
        print(f"Exportación cancelada para '{sufijo}': imagen no disponible.")
        return

    nombre_tarea = f'{PREFIJO}_{AÑO}_{sufijo}'
    tarea = ee.batch.Export.image.toDrive(
        image          = imagen,
        description    = nombre_tarea,
        folder         = 'Quila_Landsat',
        fileNamePrefix = nombre_tarea,
        region         = region,
        scale          = ESCALA,
        crs            = f'EPSG:{CRS_UTM}',
        maxPixels      = 1e13
    )
    tarea.start()
    print(f"Tarea iniciada: {nombre_tarea}")

exportar(img_seca,   'seca')
exportar(img_humeda, 'humeda')


── Temporada: Seca (03-01 → 05-31) ──
  Imágenes encontradas: 8
    · 2000-04-03
    · 2000-04-10
    · 2000-04-19
    · 2000-04-26
    · 2000-05-05
    · 2000-05-12
    · 2000-05-21
    · 2000-05-28
  Bandas resultantes: ['NDVI', 'TC_Brightness', 'TC_Greenness', 'TC_Wetness']

── Temporada: Húmeda (08-01 → 11-30) ──
  Imágenes encontradas: 11
    · 2000-08-09
    · 2000-08-16
    · 2000-08-25
    · 2000-09-01
    · 2000-09-10
    · 2000-09-17
    · 2000-09-26
    · 2000-10-03
    · 2000-10-12
    · 2000-10-19
    · 2000-11-29
  Bandas resultantes: ['NDVI', 'TC_Brightness', 'TC_Greenness', 'TC_Wetness']
Tarea iniciada: QLA_2000_seca
Tarea iniciada: QLA_2000_humeda


### Visualización de imágenes procesadas

In [ ]:
stats = img_seca.select('NDVI').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)

stats = img_seca.select('TC_Brightness').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)

stats = img_seca.select('TC_Greenness').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)

stats = img_seca.select('TC_Wetness').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)
#----------------Húmdad----------------------------------

stats = img_humeda.select('NDVI').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)

stats = img_humeda.select('TC_Brightness').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)

stats = img_humeda.select('TC_Greenness').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)

stats = img_humeda.select('TC_Wetness').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)


{'NDVI_max': 0.7884383565621054, 'NDVI_min': 0.0990759018989198}
{'TC_Brightness_max': 1.40174175375, 'TC_Brightness_min': 0.0015410202851604117}
{'TC_Greenness_max': 0.32619985800165613, 'TC_Greenness_min': -0.032339597640008634}
{'TC_Wetness_max': 0.028094597630556953, 'TC_Wetness_min': -0.4976076234577043}
{'NDVI_max': 0.8714166194374028, 'NDVI_min': 0.11782471285068663}
{'TC_Brightness_max': 1.3904340614979045, 'TC_Brightness_min': 0.0011833357088460548}
{'TC_Greenness_max': 0.8354185591929428, 'TC_Greenness_min': -0.005092878558075019}
{'TC_Wetness_max': 0.0778712499038096, 'TC_Wetness_min': -0.2689192809860701}


In [ ]:
# PARÁMETROS DE VISUALIZACIÓN

# Temporada SECA (basado en valores calculados)
VIS_SECA = {
    'NDVI': {'min': 0.10, 'max': 0.79, 'palette': ['#d73027', '#fee08b', '#1a9850']},
    'TC_Brightness': {'min': 0.00, 'max': 1.40, 'palette': ['white', 'brown']},
    'TC_Greenness': {'min': -0.03, 'max': 0.33, 'palette': ['white', 'darkgreen']},
    'TC_Wetness': {'min': -0.50, 'max': 0.03, 'palette': ['brown', 'white', 'blue']}
}

# Temporada HÚMEDA (basado en valores calculados)
VIS_HUMEDA = {
    'NDVI': {'min': 0.12, 'max': 0.87, 'palette': ['#d73027', '#fee08b', '#1a9850']},
    'TC_Brightness': {'min': 0.00, 'max': 1.39, 'palette': ['white', 'brown']},
    'TC_Greenness': {'min': -0.01, 'max': 0.84, 'palette': ['white', 'darkgreen']},
    'TC_Wetness': {'min': -0.27, 'max': 0.08, 'palette': ['brown', 'white', 'blue']}
}

# Recalcular centroide
gdf = gpd.read_file(RUTA_GEOJSON)
centroid_proj = gdf.to_crs(epsg=3857).dissolve().centroid.iloc[0]
centroid = gpd.GeoSeries([centroid_proj], crs=3857).to_crs(epsg=4326).iloc[0]

#  MAPA
mapa2 = geemap.Map()
mapa2.set_center(centroid.x, centroid.y, ZOOM)

mapa2.add_gdf(gdf, layer_name='Área de estudio', style=STYLE_AREA)

if img_seca is not None:
    mapa2.addLayer(img_seca.select('NDVI'),          VIS_SECA['NDVI'], f'{AÑO} Seca – NDVI',          shown=True)
    mapa2.addLayer(img_seca.select('TC_Brightness'), VIS_SECA['TC_Brightness'],  f'{AÑO} Seca – TC Brightness',  shown=False)
    mapa2.addLayer(img_seca.select('TC_Greenness'),  VIS_SECA['TC_Greenness'],  f'{AÑO} Seca – TC Greenness',   shown=False)
    mapa2.addLayer(img_seca.select('TC_Wetness'),    VIS_SECA['TC_Wetness'],  f'{AÑO} Seca – TC Wetness',     shown=False)

if img_humeda is not None:
    mapa2.addLayer(img_humeda.select('NDVI'),          VIS_HUMEDA['NDVI'], f'{AÑO} Húmeda – NDVI',          shown=False)
    mapa2.addLayer(img_humeda.select('TC_Brightness'), VIS_HUMEDA['TC_Brightness'],  f'{AÑO} Húmeda – TC Brightness',  shown=False)
    mapa2.addLayer(img_humeda.select('TC_Greenness'),  VIS_HUMEDA['TC_Greenness'],  f'{AÑO} Húmeda – TC Greenness',   shown=False)
    mapa2.addLayer(img_humeda.select('TC_Wetness'),    VIS_HUMEDA['TC_Wetness'],  f'{AÑO} Húmeda – TC Wetness',     shown=False)

mapa2.addLayerControl()
print("Visualización de imágenes procesadas")
mapa2